# Archive ↔ JSON demo (PyKotor)

Convert capsule-like archives (ERF, RIM, MOD, SAV, BIF) to JSON with **plaintext resource embedding** where supported, and back to binary.

- **Supported formats:** ERF, RIM, BIF (MOD/SAV use ERF layout).
- **Plaintext resource types:** GFF (and ARE, DLG, GIT, IFO, UTI, UTC, …), TLK, 2DA, LIP, SSF; others stay base64.
- **CLI:** `pykotor capsule2json <input> [--output out.json] [--key-file chitin.key]` and `pykotor json2capsule <input.json> --output <capsule>`

In [ ]:
from io import BytesIO

from pykotor.resource.formats.erf import read_erf, write_erf
from pykotor.resource.formats.erf.erf_data import ERF, ERFType
from pykotor.common.misc import ResRef
from pykotor.resource.type import ResourceType
from pykotor.tools.archive_serializer import archive_to_dict, dict_to_archive

## 1. Build a minimal ERF in memory

In [ ]:
# Create a small ERF with one TXT and one GFF-like resource (as base64 for demo)
erf = ERF(erf_type=ERFType.ERF)
erf.set_data(ResRef("hello"), ResourceType.TXT, b"Hello from ERF")
erf.set_data(ResRef("data"), ResourceType.GFF, b"\x00" * 100)  # dummy GFF bytes
buf = BytesIO()
write_erf(erf, buf)
erf_bytes = buf.getvalue()
print(f"ERF size: {len(erf_bytes)} bytes")

## 2. Convert ERF → JSON (archive_to_dict)

In [ ]:
data = archive_to_dict(erf_bytes, embed_plaintext=True)
print("Keys:", list(data.keys()))
print("Format:", data["format"])
print("Resources:", [r["resref"] for r in data["resources"]])
for r in data["resources"]:
    print(f"  - {r['resref']}.{r['restype']}: encoding={r['data_encoding']}")

## 3. Round-trip: JSON → binary → read back

In [ ]:
back_bytes, ext = dict_to_archive(data)
erf2 = read_erf(BytesIO(back_bytes))
print(f"Round-trip extension: .{ext}")
for res in erf2:
    print(f"  {res.resref}.{res.restype.extension}: {res.data[:20]!r}...")

## 4. Minimal BIF in plaintext (JSON) form

In [ ]:
import base64

minimal_bif_json = {
    "format": "bif",
    "bif_type": "BIFF",
    "resources": [
        {"resref": "a", "restype": "txt", "resname_key_index": 0, "data_encoding": "base64", "data": base64.b64encode(b"A").decode("ascii")},
        {"resref": "b", "restype": "txt", "resname_key_index": 1, "data_encoding": "base64", "data": base64.b64encode(b"B").decode("ascii")},
    ],
}
bif_bytes, _ = dict_to_archive(minimal_bif_json)
from pykotor.resource.formats.bif import read_bif
bif = read_bif(bif_bytes)
print(f"BIF resources: {[(r.resref, r.data) for r in bif.resources]}")